# A/B Testing for Income Classifier

This notebook simulates A/B testing by sending test data to the server and providing feedback.

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import requests

## Load Dataset

In [ ]:
# load dataset
df = pd.read_csv('https://raw.githubusercontent.com/pplonski/datasets-for-start/master/adult/data.csv', skipinitialspace=True)
x_cols = [c for c in df.columns if c != 'income']
# set input matrix and target column
X = df[x_cols]
y = df['income']
# show first rows of data
df.head()

## Split Data into Train and Test Sets

In [ ]:
# data split train / test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state=1234)

## Run A/B Test

Send 100 test samples to the server with `status=ab_testing` to randomly test both algorithms.

In [ ]:
for i in range(100):
    input_data = dict(X_test.iloc[i])
    # Convert numpy types to Python native types and handle NaN
    input_data = {k: None if pd.isna(v) else int(v) if isinstance(v, (np.integer, np.int64)) else float(v) if isinstance(v, (np.floating, np.float64)) else v for k, v in input_data.items()}
    target = y_test.iloc[i]
    r = requests.post("http://127.0.0.1:8000/api/v1/income_classifier/predict?status=ab_testing", json=input_data)
    response = r.json()
    # provide feedback
    requests.put("http://127.0.0.1:8000/api/v1/mlrequests/{}".format(response["request_id"]), json={"feedback": target})
    
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1} requests")

print("A/B test completed!")

## Check Results

You can now check the results at:
- http://127.0.0.1:8000/api/v1/mlrequests - to see all requests
- http://127.0.0.1:8000/api/v1/abtests - to see A/B test information